# Molecular Tokenization

## Estimated Time

- **CPU:** ~15 minutes
- **GPU:** ~15 minutes
- *Category: Short*

## Overview

This tutorial teaches you how to tokenize molecules into fragment sequences. You'll learn about different token types (grammar, elements, fragments) and how to create token vocabularies for sequence models.

## Prerequisites

- [x] GSGE installed (see [Installation Guide](../../docs/getting-started/installation.md))
- [ ] Basic Python knowledge
- [ ] Completed: None (independent module)

## Learning Objectives

By the end of this tutorial, you will be able to:
- Tokenize molecules into fragment sequences
- Understand token types (grammar, elements, fragments)
- Create token vocabularies
- Handle batch processing and padding


# Molecular Fragment Tokenization

## Overview

This tutorial demonstrates **tokenization** in GSGE - the process of converting molecular SMILES into sequences of tokens that can be used with sequence models (RNNs, Transformers, etc.).

Unlike traditional SMILES tokenization (character-level or atom-level), GSGE tokenization operates at the **fragment level**, providing:
- Semantic meaning through functional groups
- Shorter sequences (fewer tokens per molecule)
- Direct integration with learned fragment embeddings
- Grammar-aware structure encoding

## Learning Objectives

By the end of this tutorial, you will be able to:
1. Tokenize SMILES strings into fragment-based token sequences
2. Understand different token types (grammar, elements, fragments)
3. Convert tokens to numerical IDs for model input
4. Process molecules in batch with parallel tokenization
5. Access and understand the token vocabulary
6. Use tokenized sequences with PyTorch models
7. Handle padding and masking for batch processing

## Prerequisites

- GSGE installed with dependencies
- A pre-built vocabulary (or use the provided test vocabulary)
- Basic understanding of sequence models (helpful but not required)
- Familiarity with PyTorch (for the model integration section)

## Table of Contents

- [1. Setup and Imports](#1-setup-and-imports)
  
- [2. Prepare Sample Data](#2-prepare-sample-data)
  
- [3. Load GSGE Instance](#3-load-gsge-instance)
  
- [4. Single Molecule Tokenization](#4-single-molecule-tokenization)
  
- [5. Understanding Token Types](#5-understanding-token-types)
  
- [6. Token-to-ID Conversion](#6-token-to-id-conversion)
  
- [7. Batch Tokenization](#7-batch-tokenization)
  
- [8. Exploring the Vocabulary](#8-exploring-the-vocabulary)
  
- [9. Using Tokens with Sequence Models](#9-using-tokens-with-sequence-models)
  
- [10. Summary and Next Steps](#10-summary-and-next-steps)

---

## 1. Setup and Imports

Import the necessary modules:
- `GSGE`: Main class providing tokenization functionality
- `get_tests_dir`: Utility to locate test fixtures

In [9]:
# GSGE imports
from GSGE import GSGE, get_tests_dir

# For later: model integration
import torch
import numpy as np

print("[OK] Imports successful")

[OK] Imports successful


## 2. Prepare Sample Data

We'll use the same **cyclic peptide** examples from the compound graphs tutorial. These complex molecules demonstrate the power of fragment-based tokenization.

### Important: SMILES Standardization

**Always standardize SMILES** before tokenization:
- Ensures consistent fragment extraction
- Removes stereochemistry ambiguities
- Normalizes tautomers and charges

The example SMILES below are already standardized.

In [10]:
# Example cyclic peptides (already standardized)
peptide_smiles = [
    'CC(C)CC1NC(=O)CCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(Cc2ccccc2)N(C)C(=O)CNC(=O)C(C(C)O)NC(=O)C(Cc2ccccc2)NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)NC(=O)C(Cc2ccccc2)NC1=O',
    'CCC(C)C1NC(=O)C(CC(C)C)N(C)C(=O)CCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(Cc2ccccc2)N(C)C(=O)C(CC(C)C)NC(=O)CN(C)C(=O)C(C(C)O)NC(=O)C(C)N(C)C(=O)C(C(C)C)N(C)C1=O',
    'CCC(C)C1NC(=O)C(Cc2ccccc2)NC(=O)C(C)N(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(CC(C)C)NC(=O)C(Cc2ccccc2)N(C)C(=O)C(Cc2ccccc2)NC(=O)C(CC(C)C)NC(=O)C(C(C)O)NC(=O)C(CC(C)C)NC1=O',
    'CC(C)CC1NC(=O)C(CC(C)C)N(C)C(=O)C(Cc2ccccc2)N(C)C(=O)CNC(=O)C(C(C)O)NC(=O)C(CC(C)C)N(C)C(=O)C(Cc2ccccc2)NC(=O)C(C)N(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(COC(C)(C)C)NC1=O',
    'CCC(C)C1NC(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)CCCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(C(C)O)NC(=O)C(Cc2ccccc2)NC(=O)C(C(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(Cc2ccccc2)N(C)C1=O',
    'CC(C)CC1C(=O)N(C)C(Cc2ccccc2)C(=O)NC(C(C)O)C(=O)N(C)C(C)C(=O)N(C)C(CC(C)C)C(=O)N(C)C(CC(C)C)C(=O)NC(C(=O)N2CCCCC2)CC(=O)N(C)CCCC(=O)NC(COC(C)(C)C)C(=O)N(C)CC(=O)N1C',
    'CC(C)CC1C(=O)N(C)C(CC(C)C)C(=O)NC(C(=O)N(C)C(Cc2ccccc2)C(=O)NC(C)C(=O)N2CCCCC2)CC(=O)NC(C(C)C)C(=O)N(C)C(C)C(=O)N(C)C(Cc2ccccc2)C(=O)N(C)C(CC(C)C)C(=O)NC(C(C)O)C(=O)N(C)CC(=O)NC(C(C)C)C(=O)N1C',
    'CCC(C)C1NC(=O)C(C)N(C)C(=O)CC(C(=O)N(C)C(Cc2ccccc2)C(=O)NC(C)C(=O)N2CCCCC2)NC(=O)C(Cc2ccccc2)N(C)C(=O)C(C)NC(=O)C(CC(C)C)N(C)C(=O)C(C)N(C)C(=O)C(C(C)O)NC(=O)C(C(C)C)N(C)C(=O)C(C)N(C)C1=O',
    'CC(C)CC1NC(=O)C(Cc2ccccc2)N(C)C(=O)C(Cc2ccccc2)N(C)C(=O)C(CC(C)C)NC(=O)C(C)N(C)C(=O)C(C(C)O)NC(=O)C(C)N(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(CC(C)C)N(C)C1=O',
    'CC(C)CC1NC(=O)C(Cc2ccccc2)N(C)C(=O)C(CC(C)C)N(C)C(=O)CCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(CC(C)C)N(C)C(=O)C(Cc2ccccc2)N(C)C(=O)C(C(C)O)NC(=O)C(Cc2ccccc2)N(C)C1=O',
    'CC(C)CC1NC(=O)C(C)N(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(CC(C)C)N(C)C(=O)C(Cc2ccccc2)N(C)C(=O)C(CC(C)C)NC(=O)C(Cc2ccccc2)N(C)C(=O)C(C)N(C)C(=O)C(C(C)O)NC1=O',
    'CC(C)CC1C(=O)NC(C(C)O)C(=O)N(C)C(C)C(=O)N(C)C(CC(C)C)C(=O)NC(C(=O)N2CCCCC2)CC(=O)N(C)CCCC(=O)NC(C)C(=O)N(C)C(Cc2ccccc2)C(=O)NC(Cc2ccccc2)C(=O)N1C',
    'CC(C)CC1C(=O)N(C)C(Cc2ccccc2)C(=O)NCC(=O)NC(C(=O)N2CCCCC2)CC(=O)N(C)CCCC(=O)NC(C(C)O)C(=O)N(C)C(C)C(=O)N(C)C(Cc2ccccc2)C(=O)NC(COC(C)(C)C)C(=O)N1C',
    'CC(C)CC1C(=O)NC(C(C)O)C(=O)N(C)C(C)C(=O)N(C)C(CC(C)C)C(=O)N(C)C(Cc2ccccc2)C(=O)NC(COC(C)(C)C)C(=O)NC(C(=O)N2CCCCC2)CC(=O)N(C)CCCC(=O)N(C)C(C)C(=O)N1C'
]

print(f"[OK] Loaded {len(peptide_smiles)} cyclic peptide examples")
print(f"\nExample SMILES (first 80 characters):")
print(peptide_smiles[0][:80] + "...")

[OK] Loaded 14 cyclic peptide examples

Example SMILES (first 80 characters):
CC(C)CC1NC(=O)CCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(Cc2ccccc2)N(C)C(=O)CNC(=O)C(C(...


## 3. Load GSGE Instance

Load a pre-built GSGE instance with vocabulary and embeddings:

In [11]:
# Load pre-built GSGE instance from test data
tests_dir = get_tests_dir()
if tests_dir is None:
    raise RuntimeError("Cannot find tests directory. Run from GSGE source checkout.")
pkl_path = tests_dir / 'test_gsge_save_with_descriptors.pkl'

gsge = GSGE(GSGE_load_path=pkl_path)

# Get the token-to-ID vocabulary (GSGE_vocab, not GS_vocab)
gsg_vocab = gsge.get_GSGE_vocab()

print(f"[OK] GSGE loaded successfully")
print(f"  - Vocabulary size: {gsge.get_GS_vocab().num_fragments} fragments")
print(f"  - Total tokens: {len(gsg_vocab)} (including grammar and elements)")

Data loaded and restored into gsge from C:\Users\Admin\OneDrive - Universiteit Leiden\PhD\Repos\jasper\gsge\tests\test_gsge_save_with_descriptors.pkl
[OK] GSGE loaded successfully
  - Vocabulary size: 185 fragments
  - Total tokens: 232 (including grammar and elements)


## 4. Single Molecule Tokenization

Let's tokenize a single molecule to understand the process.

### What is Tokenization?

Tokenization converts a SMILES string into a **sequence of tokens**, where each token represents:
- **Grammar symbols**: Structure encoding (`:0`, `Ring1`, `Branch`, `pop`)
- **Elements**: Single atoms (`C`, `N`, `O`, `S`)
- **Fragments**: Molecular fragments from the vocabulary (`GS_frag_38`, `GS_frag_76`)

### Why Fragment-Level Tokenization?

Compared to character-level or atom-level tokenization:
- **Shorter sequences**: Fewer tokens per molecule (faster training)
- **Semantic meaning**: Each token represents a functional group
- **Learned representations**: Can use pre-trained fragment embeddings

In [12]:
# Tokenize a single molecule
test_smiles = peptide_smiles[0]

tokens = gsge.preprocess_from_SMILES(test_smiles)

print(f"[OK] Tokenization complete")
print(f"  - Original SMILES length: {len(test_smiles)} characters")
print(f"  - Token sequence length: {len(tokens)} tokens")
print(f"\nFirst 20 tokens:")
print(tokens[:20])
print(f"\n... {len(tokens) - 20} more tokens")

[OK] Tokenization complete
  - Original SMILES length: 158 characters
  - Token sequence length: 87 tokens

First 20 tokens:
[':0', 'GS_frag_38', 'C', ':0', 'GS_frag_76', 'Ring1', ':3', 'GS_frag_107', 'O', ':3', 'GS_frag_22', 'Ring1', ':0', 'GS_frag_69', 'pop', 'Ring1', ':0', 'GS_frag_76', 'Ring1', ':6']

... 67 more tokens


### Understanding the Output:

Each token represents part of the molecular structure:
```python
[':0',           # Grammar: bond type
 'GS_frag_38',   # Fragment: amino acid residue
 'C',            # Element: single carbon
 ':0',           # Grammar: bond type
 'GS_frag_76',   # Fragment: peptide bond
 'Ring1',        # Grammar: ring closure
 ...]
```

## 5. Understanding Token Types

GSGE uses three types of tokens:

### 1. Grammar Tokens

Encode molecular structure (from Group-SELFIES):
- **Bond types**: `:0`, `:1`, `:2`, etc. (single, double, triple, aromatic)
- **Ring closures**: `Ring1`, `Ring2`, `Ring3`
- **Branching**: `Branch`, `=Branch`, `#Branch`
- **Stack operations**: `pop` (close branch)

### 2. Element Tokens

Single atoms not part of larger fragments:
- `C`, `N`, `O`, `S`, `F`, `Cl`, `Br`, `I`, etc.

### 3. Fragment Tokens

Molecular fragments from the vocabulary:
- `GS_frag_0`, `GS_frag_1`, ..., `GS_frag_N`
- Each represents a specific functional group or substructure

### Special Tokens

For model training:
- `[PAD]`: Padding token (ID 0)
- `[UNK]`: Unknown token (ID 1)
- `[CLS]`: Classification token (ID 2)
- `[SEP]`: Separator token (ID 3)
- `[MASK]`: Masking token (ID 4)

In [13]:
# Count different token types
grammar_tokens = [t for t in tokens if t.startswith(':') or t in ['Ring1', 'Ring2', 'Ring3', 'Branch', '=Branch', '#Branch', 'pop', '->']]
element_tokens = [t for t in tokens if t in ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P']]
fragment_tokens = [t for t in tokens if t.startswith('GS_frag_')]

print(f"Token type distribution:")
print(f"  - Grammar tokens: {len(grammar_tokens)} ({len(grammar_tokens)/len(tokens)*100:.1f}%)")
print(f"  - Element tokens: {len(element_tokens)} ({len(element_tokens)/len(tokens)*100:.1f}%)")
print(f"  - Fragment tokens: {len(fragment_tokens)} ({len(fragment_tokens)/len(tokens)*100:.1f}%)")
print(f"\nExample fragment tokens:")
print(fragment_tokens[:10])

Token type distribution:
  - Grammar tokens: 63 (72.4%)
  - Element tokens: 3 (3.4%)
  - Fragment tokens: 21 (24.1%)

Example fragment tokens:
['GS_frag_38', 'GS_frag_76', 'GS_frag_107', 'GS_frag_22', 'GS_frag_69', 'GS_frag_76', 'GS_frag_107', 'GS_frag_108', 'GS_frag_77', 'GS_frag_108']


## 6. Token-to-ID Conversion

Machine learning models require **numerical input**, not strings. Each token is mapped to a unique integer ID.

### Token Vocabulary

The vocabulary is a dictionary mapping token strings to integer IDs:
```python
{
    '[PAD]': 0,
    '[UNK]': 1,
    ':0': 13,
    'C': 29,
    'GS_frag_38': 85,
    ...
}
```

In [14]:
# Get the token vocabulary (token-to-ID mapping)
token_vocab = gsge.get_GSGE_vocab()

print(f"[OK] Token vocabulary loaded")
print(f"  - Total tokens: {len(token_vocab)}")
print(f"\nFirst 20 token mappings:")
for i, (token, idx) in enumerate(list(token_vocab.items())[:20]):
    print(f"  {token:15s} -> {idx}")

[OK] Token vocabulary loaded
  - Total tokens: 232

First 20 token mappings:
  [PAD]           -> 0
  [UNK]           -> 1
  [CLS]           -> 2
  [SEP]           -> 3
  [MASK]          -> 4
  ->              -> 5
  pop             -> 6
  Ring1           -> 7
  Ring2           -> 8
  Ring3           -> 9
  Branch          -> 10
  =Branch         -> 11
  #Branch         -> 12
  :0              -> 13
  :1              -> 14
  :2              -> 15
  :3              -> 16
  :4              -> 17
  :5              -> 18
  :6              -> 19


### Converting Tokens to IDs

Use the vocabulary to convert token sequences to ID sequences:

In [15]:
# Manual conversion (for understanding)
token_ids_manual = [token_vocab[token] for token in tokens]

print(f"[OK] Tokens converted to IDs")
print(f"  - Sequence length: {len(token_ids_manual)}")
print(f"\nFirst 20 token IDs:")
print(token_ids_manual[:20])

# Show mapping for first few tokens
print(f"\nToken -> ID mapping (first 10):")
for token, token_id in zip(tokens[:10], token_ids_manual[:10]):
    print(f"  {token:15s} -> {token_id}")

[OK] Tokens converted to IDs
  - Sequence length: 87

First 20 token IDs:
[13, 85, 29, 13, 123, 7, 16, 154, 31, 16, 69, 7, 13, 116, 6, 7, 13, 123, 7, 19]

Token -> ID mapping (first 10):
  :0              -> 13
  GS_frag_38      -> 85
  C               -> 29
  :0              -> 13
  GS_frag_76      -> 123
  Ring1           -> 7
  :3              -> 16
  GS_frag_107     -> 154
  O               -> 31
  :3              -> 16


## 7. Batch Tokenization

For training, we need to tokenize many molecules efficiently. GSGE provides **parallel batch tokenization**.

### Batch Processing Features:

- **Parallel processing**: Uses multiple workers for speed
- **Automatic padding**: Pads sequences to the same length (using 0s for padding)
- **Numpy arrays**: Returns ready-to-use 2D array for PyTorch/TensorFlow

### What `parallel_tokenize_SMILES_list()` Returns:

```python
padded_results, smiles_list = gsge.parallel_tokenize_SMILES_list(smiles_list)
# padded_results: 2D numpy array of shape (n_molecules, max_seq_len) with token IDs
# smiles_list: Original SMILES strings (for reference)
```

To get actual sequence lengths (before padding), count non-zero tokens:
```python
seq_lengths = [np.count_nonzero(row) for row in padded_results]
```

In [16]:
# Batch tokenize all peptides
tokenized_batch = gsge.parallel_tokenize_SMILES_list(peptide_smiles)

# The function returns:
# - [0]: padded_results: 2D numpy array of token IDs (shape: [n_molecules, max_seq_len])
# - [1]: smiles_list: original SMILES strings
padded_results = tokenized_batch[0]  # 2D numpy array with padded token IDs
smiles_list = tokenized_batch[1]     # Original SMILES strings

# Compute actual sequence lengths (count non-zero tokens, since padding uses 0)
seq_lengths = [np.count_nonzero(row) for row in padded_results]

print(f"[OK] Batch tokenization complete")
print(f"  - Number of molecules: {len(padded_results)}")
print(f"  - Padded sequence length: {padded_results.shape[1]}")
print(f"\nSequence length statistics:")
print(f"  - Min length: {min(seq_lengths)}")
print(f"  - Max length: {max(seq_lengths)}")
print(f"  - Average length: {np.mean(seq_lengths):.1f}")

parallel_tokenize_batch: 100%|██████████| 14/14 [01:16<00:00,  5.48s/it]


[OK] Batch tokenization complete
  - Number of molecules: 14
  - Padded sequence length: 119

Sequence length statistics:
  - Min length: 79
  - Max length: 119
  - Average length: 95.9


### Understanding Padding:

Sequences are padded with `[PAD]` tokens (ID 0) to make them the same length:

```python
# Original sequences (different lengths)
seq1 = [13, 85, 29, ...]  # 85 tokens
seq2 = [13, 76, 107, ...]  # 92 tokens

# After padding (same length)
seq1_padded = [13, 85, 29, ..., 0, 0, 0]  # 120 tokens (35 padding)
seq2_padded = [13, 76, 107, ..., 0, 0, 0]  # 120 tokens (28 padding)
```

In [17]:
# Examine the first tokenized sequence
first_seq = padded_results[0]  # First row from the 2D array
first_len = seq_lengths[0]
first_smiles = smiles_list[0]

print(f"First sequence details:")
print(f"  - SMILES (first 60 chars): {first_smiles[:60]}...")
print(f"  - Original length: {first_len} tokens")
print(f"  - Padded length: {len(first_seq)} tokens")
print(f"  - Padding tokens: {len(first_seq) - first_len}")
print(f"\nFirst 20 token IDs:")
print(first_seq[:20])
print(f"\nLast 20 token IDs (should be mostly 0s):")
print(first_seq[-20:])

First sequence details:
  - SMILES (first 60 chars): CC(C)CC1NC(=O)CCN(C)C(=O)CC(C(=O)N2CCCCC2)NC(=O)C(Cc2ccccc2)...
  - Original length: 87 tokens
  - Padded length: 119 tokens
  - Padding tokens: 32

First 20 token IDs:
[ 13  85  29  13 123   7  16 154  31  16  69   7  13 116   6   7  13 123
   7  19]

Last 20 token IDs (should be mostly 0s):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## 8. Exploring the Vocabulary

Understanding the vocabulary structure helps with model design and debugging.

In [18]:
# Analyze vocabulary composition
# Use GSGE_vocab for the token-to-ID dictionary
token_vocab = gsge.get_GSGE_vocab()

# Count token types
special_tokens = [t for t in token_vocab.keys() if t.startswith('[')]
grammar_tokens = [t for t in token_vocab.keys() if t.startswith(':') or t in ['Ring1', 'Ring2', 'Ring3', 'Branch', '=Branch', '#Branch', 'pop', '->']]
element_tokens = [t for t in token_vocab.keys() if t in ['B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'Br', 'I']]
fragment_tokens = [t for t in token_vocab.keys() if t.startswith('GS_frag_')]

print(f"Vocabulary composition:")
print(f"  - Special tokens: {len(special_tokens)}")
print(f"  - Grammar tokens: {len(grammar_tokens)}")
print(f"  - Element tokens: {len(element_tokens)}")
print(f"  - Fragment tokens: {len(fragment_tokens)}")
print(f"  - Total: {len(token_vocab)}")

print(f"\nSpecial tokens:")
for token in special_tokens:
    print(f"  {token:10s} -> ID {token_vocab[token]}")

print(f"\nElement tokens:")
for token in element_tokens:
    print(f"  {token:10s} -> ID {token_vocab[token]}")

Vocabulary composition:
  - Special tokens: 5
  - Grammar tokens: 23
  - Element tokens: 11
  - Fragment tokens: 185
  - Total: 232

Special tokens:
  [PAD]      -> ID 0
  [UNK]      -> ID 1
  [CLS]      -> ID 2
  [SEP]      -> ID 3
  [MASK]     -> ID 4

Element tokens:
  B          -> ID 28
  C          -> ID 29
  N          -> ID 30
  O          -> ID 31
  F          -> ID 32
  Si         -> ID 33
  P          -> ID 34
  S          -> ID 35
  Cl         -> ID 36
  Br         -> ID 40
  I          -> ID 44


## 9. Using Tokens with Sequence Models

Tokenized sequences can be used with RNNs, Transformers, or other sequence models.

### Example: Preparing Data for PyTorch

Here's a complete example of preparing tokenized data for model training:

In [19]:
# Convert to PyTorch tensors
token_ids_tensor = torch.LongTensor(padded_results)  # Already a 2D numpy array
seq_lengths_tensor = torch.LongTensor(seq_lengths)

# Create attention masks (1 for real tokens, 0 for padding)
attention_masks = torch.zeros_like(token_ids_tensor)
for i, length in enumerate(seq_lengths):
    attention_masks[i, :length] = 1

print(f"[OK] Data prepared for PyTorch")
print(f"  - Token IDs shape: {token_ids_tensor.shape}")
print(f"  - Attention masks shape: {attention_masks.shape}")
print(f"  - Sequence lengths shape: {seq_lengths_tensor.shape}")

# Example: first sequence
print(f"\nFirst sequence:")
print(f"  - Token IDs: {token_ids_tensor[0][:20]}")
print(f"  - Attention mask: {attention_masks[0][:20]}")
print(f"  - Original length: {seq_lengths_tensor[0]}")

[OK] Data prepared for PyTorch
  - Token IDs shape: torch.Size([14, 119])
  - Attention masks shape: torch.Size([14, 119])
  - Sequence lengths shape: torch.Size([14])

First sequence:
  - Token IDs: tensor([ 13,  85,  29,  13, 123,   7,  16, 154,  31,  16,  69,   7,  13, 116,
          6,   7,  13, 123,   7,  19])
  - Attention mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
  - Original length: 87


### Example: RNN Model with Fragment Embeddings

Here's a simple RNN model that uses fragment embeddings:

In [20]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class MolecularRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        # Token embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )
        
        # Classification head
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
    
    def forward(self, token_ids, seq_lengths):
        # Embed tokens
        embedded = self.embedding(token_ids)  # [batch, seq_len, emb_dim]
        
        # Pack sequences for efficient processing
        packed = pack_padded_sequence(
            embedded, 
            seq_lengths.cpu(), 
            batch_first=True, 
            enforce_sorted=False
        )
        
        # LSTM forward pass
        packed_output, (hidden, cell) = self.lstm(packed)
        
        # Concatenate final forward and backward hidden states
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # [batch, hidden*2]
        
        # Classification
        output = self.fc(final_hidden)
        return output

# Create model
model = MolecularRNN(
    vocab_size=len(token_vocab),
    embedding_dim=128,
    hidden_dim=256,
    num_classes=1  # For regression; use num_classes for classification
)

print(f"[OK] RNN model created")
print(f"  - Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
with torch.no_grad():
    output = model(token_ids_tensor, seq_lengths_tensor)
    print(f"  - Output shape: {output.shape}")

[OK] RNN model created
  - Parameters: 2,397,697
  - Output shape: torch.Size([14, 1])


### Using Pre-trained Fragment Embeddings

If you've trained a GAE, you can initialize the embedding layer with learned fragment embeddings:

In [21]:
# Load pre-trained fragment embeddings
fragment_embeddings = gsge.get_fragment_embeddings()

if fragment_embeddings is not None:
    print(f"[OK] Pre-trained fragment embeddings available")
    print(f"  - Shape: {fragment_embeddings.shape}")
    print(f"  - Dimension: {fragment_embeddings.shape[1]}")
    
    # Note: Fragment embeddings only cover fragment tokens
    # You need to combine them with embeddings for grammar/element tokens
    
    print(f"\nTo use pre-trained embeddings:")
    print(f"1. Create embedding matrix with shape [{len(token_vocab)}, {fragment_embeddings.shape[1]}]")
    print(f"2. Initialize fragment token embeddings with pre-trained values")
    print(f"3. Randomly initialize grammar and element token embeddings")
    print(f"4. Optionally freeze fragment embeddings during training")
else:
    print("[!] No pre-trained embeddings available")
    print("  Train embeddings from scratch or run GAE training first")

[OK] Pre-trained fragment embeddings available
  - Shape: torch.Size([185, 128])
  - Dimension: 128

To use pre-trained embeddings:
1. Create embedding matrix with shape [232, 128]
2. Initialize fragment token embeddings with pre-trained values
3. Randomly initialize grammar and element token embeddings
4. Optionally freeze fragment embeddings during training


### Example: Transformer Model

Transformers can also use tokenized sequences:

In [22]:
class MolecularTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super().__init__()
        # Token embedding
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        # Positional encoding
        self.pos_encoder = nn.Embedding(512, d_model)  # Max seq len = 512
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, token_ids, attention_mask):
        batch_size, seq_len = token_ids.shape
        
        # Token embeddings + positional embeddings
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand(batch_size, -1)
        embedded = self.embedding(token_ids) + self.pos_encoder(positions)
        
        # Create padding mask (True for padding tokens)
        padding_mask = (attention_mask == 0)
        
        # Transformer forward pass
        encoded = self.transformer(embedded, src_key_padding_mask=padding_mask)
        
        # Use [CLS] token (first token) for classification
        cls_output = encoded[:, 0, :]  # [batch, d_model]
        
        # Classification
        output = self.fc(cls_output)
        return output

# Create Transformer model
transformer_model = MolecularTransformer(
    vocab_size=len(token_vocab),
    d_model=256,
    nhead=8,
    num_layers=4,
    num_classes=1
)

print(f"[OK] Transformer model created")
print(f"  - Parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")

# Test forward pass
with torch.no_grad():
    output = transformer_model(token_ids_tensor, attention_masks)
    print(f"  - Output shape: {output.shape}")

[OK] Transformer model created
  - Parameters: 3,349,761
  - Output shape: torch.Size([14, 1])


## 10. Summary and Next Steps

### What You Learned:

[OK] **Fragment-based tokenization**: Convert SMILES to semantic token sequences

[OK] **Token types**: Grammar, elements, and fragments each serve different purposes

[OK] **Token vocabularies**: Map tokens to numerical IDs for model input

[OK] **Batch processing**: Efficiently tokenize multiple molecules with padding

[OK] **PyTorch integration**: Prepare data for RNN and Transformer models

[OK] **Pre-trained embeddings**: Leverage learned fragment representations

### Key Takeaways:

1. **Fragment-level tokenization** is more semantic than character/atom-level
2. **Shorter sequences** mean faster training and inference
3. **Pre-trained embeddings** (from GAE) provide better initialization
4. **Padding and masking** are essential for batch processing
5. **Attention mechanisms** can focus on important fragments

## Next Steps

Now that you understand tokenization, explore:

1. **Train GAE**: Learn fragment embeddings from your data
   - See: [`../03_GAE/README.md`](../03_GAE/README.md)

2. **Use Embeddings**: Apply learned embeddings in sequence models
   - See: [`../04_use_embeddings/README.md`](../04_use_embeddings/README.md)

3. **Build Vocabularies**: Create custom vocabularies for your chemical space
   - See: [`../00_making_vocabs/README.md`](../00_making_vocabs/README.md)

4. **Molecular Generation**: Train sequence-to-sequence models for generation

5. **Property Prediction**: Train RNN/Transformer models for prediction tasks

## Additional Resources

- [Tutorial README](README.md) - Overview and quick start guide
- [Group-SELFIES Paper](https://doi.org/10.1039/D3DD00012E) - Original tokenization method
- [PyTorch RNN Tutorial](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html)
- [Transformers Documentation](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)
- [GSGE Documentation](../../docs/index.md) - Complete package documentation

---

**Questions or issues?** Open an issue on [GitHub](https://github.com/CDDLeiden/GSGE/issues)